In [ ]:
# 1. Get embedding
import re
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
SEQ = "MRWCLLLIWAQGLRQAPLASGMMTGTIETTGNISAEKGGSIILQCHLSSTTAQVTQVNWEQQDQLLAICNADLGWHISPSFKDRVAPGPGLGLTLQSLTVNDTGEYFCIYHTYPDGTYTGRIFLEVLESSVAEHGARFQIPLLGAMAATLVVICTAVIVVVALTRKKKALRIHSVEGDLRRKSAGQEEWSPSAPSPPGSCVQAEAAPAGLCGEQRGEDCAELHDYFNVLSYRSLGNCSFFTETG"

In [ ]:

tokenizer = T5Tokenizer.from_pretrained('Rostlab/prot_t5_xl_uniref50', local_files_only=True)
emb_model = T5ForConditionalGeneration.from_pretrained('Rostlab/prot_t5_xl_uniref50', local_files_only=True)
emb_model = emb_model.eval()
emb_model = emb_model.to(device)

In [3]:
with torch.no_grad():
    sequences_Example = [" ".join(SEQ)]
    sequences_Example = [re.sub(r"[UZOB]", "X", sequence) for sequence in sequences_Example]
    input_seq = sequences_Example[0]
    tokens = tokenizer(input_seq, add_special_tokens=True, padding=True, return_tensors="pt")
    tokens = tokens.to(device)
    encoder_outputs = emb_model.encoder(
        input_ids=tokens.input_ids,
        attention_mask=tokens.attention_mask
    )
    embedding = encoder_outputs.last_hidden_state[0]
    embedding = embedding.cpu().numpy()
    print(embedding.shape, len(SEQ))
    # torch.save(embedding, "./data/seq_emb/Q495A1.pt")

(245, 1024) 244


In [ ]:
#2. Get Geometric Feature
from Bio.PDB import PDBParser
from Bio import pairwise2
from tqdm import tqdm
import pandas as pd
import numpy as np
import warnings
import requests
import shutil
import pickle
import torch
import time
import os
import re
import gc
pdb_parser = PDBParser(QUIET=True)
DSSP = "./data/dssp-2.0.4/mkdssp"
SEQ = "MRWCLLLIWAQGLRQAPLASGMMTGTIETTGNISAEKGGSIILQCHLSSTTAQVTQVNWEQQDQLLAICNADLGWHISPSFKDRVAPGPGLGLTLQSLTVNDTGEYFCIYHTYPDGTYTGRIFLEVLESSVAEHGARFQIPLLGAMAATLVVICTAVIVVVALTRKKKALRIHSVEGDLRRKSAGQEEWSPSAPSPPGSCVQAEAAPAGLCGEQRGEDCAELHDYFNVLSYRSLGNCSFFTETG"

In [5]:
d3to1 = {
    'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
    'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N', 
    'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W', 
    'ALA': 'A', 'VAL':'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'
}

non_standard_residue_substitutions = {
    '2AS':'ASP', '3AH':'HIS', '5HP':'GLU', 'ACL':'ARG', 'AGM':'ARG', 'AIB':'ALA', 'ALM':'ALA', 'ALO':'THR', 'ALY':'LYS', 'ARM':'ARG',
    'ASA':'ASP', 'ASB':'ASP', 'ASK':'ASP', 'ASL':'ASP', 'ASQ':'ASP', 'AYA':'ALA', 'BCS':'CYS', 'BHD':'ASP', 'BMT':'THR', 'BNN':'ALA',
    'BUC':'CYS', 'BUG':'LEU', 'C5C':'CYS', 'C6C':'CYS', 'CAS':'CYS', 'CCS':'CYS', 'CEA':'CYS', 'CGU':'GLU', 'CHG':'ALA', 'CLE':'LEU', 'CME':'CYS',
    'CSD':'ALA', 'CSO':'CYS', 'CSP':'CYS', 'CSS':'CYS', 'CSW':'CYS', 'CSX':'CYS', 'CXM':'MET', 'CY1':'CYS', 'CY3':'CYS', 'CYG':'CYS',
    'CYM':'CYS', 'CYQ':'CYS', 'DAH':'PHE', 'DAL':'ALA', 'DAR':'ARG', 'DAS':'ASP', 'DCY':'CYS', 'DGL':'GLU', 'DGN':'GLN', 'DHA':'ALA',
    'DHI':'HIS', 'DIL':'ILE', 'DIV':'VAL', 'DLE':'LEU', 'DLY':'LYS', 'DNP':'ALA', 'DPN':'PHE', 'DPR':'PRO', 'DSN':'SER', 'DSP':'ASP',
    'DTH':'THR', 'DTR':'TRP', 'DTY':'TYR', 'DVA':'VAL', 'EFC':'CYS', 'FLA':'ALA', 'FME':'MET', 'GGL':'GLU', 'GL3':'GLY', 'GLZ':'GLY',
    'GMA':'GLU', 'GSC':'GLY', 'HAC':'ALA', 'HAR':'ARG', 'HIC':'HIS', 'HIP':'HIS', 'HMR':'ARG', 'HPQ':'PHE', 'HTR':'TRP', 'HYP':'PRO',
    'IAS':'ASP', 'IIL':'ILE', 'IYR':'TYR', 'KCX':'LYS', 'LLP':'LYS', 'LLY':'LYS', 'LTR':'TRP', 'LYM':'LYS', 'LYZ':'LYS', 'MAA':'ALA', 'MEN':'ASN',
    'MHS':'HIS', 'MIS':'SER', 'MLE':'LEU', 'MPQ':'GLY', 'MSA':'GLY', 'MSE':'MET', 'MVA':'VAL', 'NEM':'HIS', 'NEP':'HIS', 'NLE':'LEU',
    'NLN':'LEU', 'NLP':'LEU', 'NMC':'GLY', 'OAS':'SER', 'OCS':'CYS', 'OMT':'MET', 'PAQ':'TYR', 'PCA':'GLU', 'PEC':'CYS', 'PHI':'PHE',
    'PHL':'PHE', 'PR3':'CYS', 'PRR':'ALA', 'PTR':'TYR', 'PYX':'CYS', 'SAC':'SER', 'SAR':'GLY', 'SCH':'CYS', 'SCS':'CYS', 'SCY':'CYS',
    'SEL':'SER', 'SEP':'SER', 'SET':'SER', 'SHC':'CYS', 'SHR':'LYS', 'SMC':'CYS', 'SOC':'CYS', 'STY':'TYR', 'SVA':'SER', 'TIH':'ALA',
    'TPL':'TRP', 'TPO':'THR', 'TPQ':'ALA', 'TRG':'LYS', 'TRO':'TRP', 'TYB':'TYR', 'TYI':'TYR', 'TYQ':'TYR', 'TYS':'TYR', 'TYY':'TYR',
    # Map to Self
    'CYS':'CYS',
    'ASP':'ASP',
    'SER':'SER',
    'GLN':'GLN',
    'LYS':'LYS',
    'ILE':'ILE',
    'PRO':'PRO',
    'THR':'THR',
    'PHE':'PHE',
    'ASN':'ASN',
    'GLY':'GLY',
    'HIS':'HIS',
    'LEU':'LEU',
    'ARG':'ARG',
    'TRP':'TRP',
    'ALA':'ALA',
    'VAL':'VAL',
    'GLU':'GLU',
    'TYR':'TYR',
    'MET':'MET'
}

In [6]:
def get_pdb_seq(pdb_file):
    pdb_structure = pdb_parser.get_structure("peptide", pdb_file)[0]["A"]
    aa_seq = []
    for r in pdb_structure.get_list():
        res_name = r.get_resname()
        if(res_name not in d3to1.keys()):
        # if(res_name not in d3to1.keys()):
            aa_seq.append("X")
        else:
            aa_seq.append(d3to1[
                non_standard_residue_substitutions[res_name]])
    aa_seq = "".join(aa_seq)
    return aa_seq

def get_pdb_xyz(pdb_file):
    """
    get the coordinates
    """
    current_pos = -1000
    X = []
    current_aa = {}  # N, CA, C, O, R
    with open(pdb_file, "r") as f:
        lines = f.readlines()
    for line in lines:
        # print(line)
        if (
            line[0:4].strip() == "ATOM" and int(line[22:26].strip()) != current_pos
        ) or line[0:4].strip() == "TER":
            if current_aa != {}:
                R_group = []
                for atom in current_aa:
                    if atom not in ["N", "CA", "C", "O"]:
                        R_group.append(current_aa[atom])
                if R_group == []:
                    R_group = [current_aa["CA"]]
                R_group = np.array(R_group).mean(0)
                X.append(
                    [
                        current_aa["N"],
                        current_aa["CA"],
                        current_aa["C"],
                        current_aa["O"],
                        R_group,
                    ]
                )
                current_aa = {}
            if line[0:4].strip() != "TER":
                current_pos = int(line[22:26].strip())

        if line[0:4].strip() == "ATOM":
            atom = line[13:16].strip()
            if atom != "H":
                xyz = np.array(
                    [line[30:38].strip(), line[38:46].strip(), line[46:54].strip()]
                ).astype(np.float32)
                current_aa[atom] = xyz
    return np.array(X)

def match_dssp(seq, dssp, ref_seq):
    alignments = pairwise2.align.globalxx(ref_seq, seq)
    ref_seq = alignments[0].seqA
    seq = alignments[0].seqB
    padded_item = np.zeros(9)

    new_dssp = []
    for aa in seq:
        if aa == "-":
            new_dssp.append(padded_item)
        else:
            new_dssp.append(dssp.pop(0))

    matched_dssp = []
    for i in range(len(ref_seq)):
        if ref_seq[i] == "-":
            continue
        matched_dssp.append(new_dssp[i])

    return matched_dssp

def process_dssp(dssp_file):
    aa_type = "ACDEFGHIKLMNPQRSTVWY"
    SS_type = "HBEGITSC"
    rASA_std = [
        115,
        135,
        150,
        190,
        210,
        75,
        195,
        175,
        200,
        170,
        185,
        160,
        145,
        180,
        225,
        115,
        140,
        155,
        255,
        230,
    ]

    with open(dssp_file, "r") as f:
        lines = f.readlines()

    seq = ""
    dssp_feature = []

    p = 0
    while lines[p].strip()[0] != "#":
        p += 1
    for i in range(p + 1, len(lines)):
        aa = lines[i][13]
        if aa == "!" or aa == "*":
            continue
        seq += aa
        SS = lines[i][16]
        if SS == " ":
            SS = "C"
        SS_vec = np.zeros(8)
        SS_vec[SS_type.find(SS)] = 1
        ACC = float(lines[i][34:38].strip())
        ASA = min(1, ACC / rASA_std[aa_type.find(aa)])
        dssp_feature.append(np.concatenate((np.array([ASA]), SS_vec)))

    return seq, dssp_feature

In [7]:
structure_path = "./data/structure/3Q0H_A.pdb"
dssp_path = "./data/dssp/3Q0H_A.dssp"
pdb_xyz = get_pdb_xyz(structure_path)
pdb_seq = get_pdb_seq(structure_path)
assert len(pdb_xyz) == len(pdb_seq)

os.system(
    f"{DSSP} -i {structure_path} -o {dssp_path}"
)
dssp_seq, dssp_matrix = process_dssp(dssp_path)
assert dssp_seq==pdb_seq

dssp_matrix = np.stack(dssp_matrix)
assert len(pdb_xyz) == dssp_matrix.shape[0]

torch.save(dssp_matrix, "./data/dssp/3Q0H_A_receptor.pt")
torch.save(pdb_xyz, "./data/structure/3Q0H_A_receptor.pt")

In [8]:
def match_emb(seq, emb, ref_seq):
    alignments = pairwise2.align.globalxx(ref_seq, seq)
    ref_seq = alignments[0].seqA
    seq = alignments[0].seqB
    padded_item = np.zeros((1024))
    new_emb = []
    count = 0
    for aa in seq:
        if aa == "-":
            new_emb.append(padded_item)
        else:
            new_emb.append(emb[count])
            count += 1
    matched_emb = []
    for i in range(len(ref_seq)):
        if ref_seq[i] == "-":
            continue
        matched_emb.append(new_emb[i])

    return matched_emb

def match_mask(seq, mask, ref_seq):
    alignments = pairwise2.align.globalxx(ref_seq, seq)
    ref_seq = alignments[0].seqA
    seq = alignments[0].seqB
    padded_item = 0.0
    new_mask = []
    count = 0
    for aa in seq:
        if aa == "-":
            new_mask.append(padded_item)
        else:
            new_mask.append(mask[count])
            count += 1
    matched_mask = []
    for i in range(len(ref_seq)):
        if ref_seq[i] == "-":
            continue
        matched_mask.append(new_mask[i])
    return matched_mask

In [9]:
emb = torch.load("./data/seq_emb/Q495A1.pt", map_location=torch.device('cpu'))
emb_matched = np.stack(match_emb(SEQ, emb, pdb_seq))
pocket_mask = np.zeros(len(pdb_seq))
pocket_mask[46:54] = 1
res = {
    "receptor_seq": pdb_seq,
    "receptor_emb": emb_matched,
    "ligand_seq": "",
    "ligand_emb": np.zeros((15, 1024)),
    "pocket_mask": pocket_mask
}
torch.save(res, "./data/seq_emb/3Q0H_A.pt")

In [10]:
# 3. Start Sampling
from dataset import *
from diff_utils import *
from model import PepEDiff
from transformers import T5Config
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch
from lightning_fabric.utilities.seed import seed_everything

from typing import Tuple
from tqdm import tqdm
GEN_NUM = 100
MODEL_PATH = "./model_diff.pt"
OUTPUT_FOLDER = "./data/sample_res/sample_TIGIT"

In [11]:
CONFIG = dict(
    # Computation Resource
    gpu_id=[0],
    num_thread=16,
    loader_num_workers=16,
    loader_prefetch_factor=4,
    # For Training
    lr=5e-5,
    l2_lambda=0.1,
    batch_size=32,
    random_seed=0,
    min_epochs=1,
    max_epochs=500,
    gradient_clip=1.0,
    lr_scheduler="LinearWarmup",
    data_noise_ration=0.1,
    augment_eps=0.15,
    max_seq_len=128,
    diff_timestep=1000,
    diff_step_size=1,
    # For Model
    node_input_dim=20 + 184,  # precalculated + generated
    edge_input_dim=450,
    d_model=1024,
    d_ff=2048,
    num_layers=2,
    num_heads=4,
    dropout_rate=0.1,
    feed_forward_proj="gated-gelu",
)


def load_model() -> PepEDiff:
    attn_config = T5Config(
        d_ff=CONFIG["d_ff"],
        d_model=CONFIG["d_model"],
        num_heads=CONFIG["num_heads"],
        num_layers=CONFIG["num_layers"],
        dropout_rate=CONFIG["dropout_rate"],
        feed_forward_proj=CONFIG["feed_forward_proj"],
        is_decoder=False,
        use_cache=False,
        is_encoder_decoder=False,
    )
    model = PepEDiff(
        attn_config,
        CONFIG["node_input_dim"],
        CONFIG["edge_input_dim"],
        CONFIG["num_layers"],
        CONFIG["augment_eps"],
    )
    model.load_state_dict(torch.load(MODEL_PATH))
    model = model.to(f"cuda:{CONFIG['gpu_id'][0]}").eval()
    return model


@torch.no_grad()
def p_sample(
    model: PepEDiff,
    noised_emb: torch.Tensor,
    batch,
    timestep: int,
    betas: torch.Tensor,
) -> torch.Tensor:
    device = next(model.parameters()).device
    if timestep <= 1:  # skip timestep=0
        return noised_emb, noised_emb
    alpha_beta_values = compute_alphas(betas)
    sqrt_recip_alphas = 1.0 / torch.sqrt(alpha_beta_values["alphas"])
    sqrt_recip_alphas_t = sqrt_recip_alphas[timestep]
    betas_t = betas[timestep]
    sqrt_one_minus_alphas_cumprod_t = alpha_beta_values[
        "sqrt_one_minus_alphas_cumprod"
    ][timestep]
    batch_size = len(batch.batch.unique())
    pocket_mask = to_dense_batch(batch.pocket_mask, batch.batch)[0]
    noised_emb = noised_emb.reshape(batch_size, CONFIG["max_seq_len"], -1)
    emb_mask = batch.peptide_mask.reshape(batch_size, CONFIG["max_seq_len"], -1)
    timesteps = torch.full(
        (noised_emb.shape[0],), timestep, device=device, dtype=torch.long
    )
    predicted_noise = model(
        # Receptor Structure Features
        coords=batch.X,
        edge_index=batch.edge_index,
        structure_feat=batch.receptor_structure,
        # Receptor Sequence Features
        seq_feat=batch.receptor_seq_emb,
        pocket_mask=pocket_mask,
        # Peptide Features
        noised_emb=noised_emb,
        emb_mask=emb_mask,
        # Others
        timesteps=timesteps,
        batch_id=batch.batch,
    )
    model_mean = sqrt_recip_alphas_t * (
        noised_emb - betas_t * predicted_noise / sqrt_one_minus_alphas_cumprod_t
    )
    posterior_variance_t = alpha_beta_values["posterior_variance"][timestep]
    noise = torch.randn_like(noised_emb)
    return model_mean + torch.sqrt(posterior_variance_t) * noise, predicted_noise


@torch.no_grad()
def p_sample_loop(
    model: PepEDiff,
    batch,
    betas: torch.Tensor,
) -> torch.Tensor:
    """
    Returns a tensor of shape (timesteps, batch_size, seq_len, n_ft)
    """
    device = next(model.parameters()).device
    batch = batch.to(device)
    ligand_emb_noise = torch.randn_like(batch.known_noise, device=device)
    print("init noise:", ligand_emb_noise.max(), ligand_emb_noise.min())
    b = ligand_emb_noise.shape[0]
    noises = []
    tqdm_bar = tqdm(
        reversed(range(0, CONFIG["diff_timestep"], CONFIG["diff_step_size"])),
        desc="sampling loop time step",
        total=int(CONFIG["diff_timestep"] / CONFIG["diff_step_size"]),
        dynamic_ncols=True,
    )
    for i in tqdm_bar:
        # Shape is (batch, seq_len, 1024)
        ligand_emb_noise, pred_noise = p_sample(
            model=model,
            noised_emb=ligand_emb_noise,
            batch=batch,
            timestep=i,
            betas=betas,
        )
        tqdm_bar.set_postfix(
            emb_min=float(ligand_emb_noise.min()),
            emb_max=float(ligand_emb_noise.max()),
            pred_min=float(pred_noise.min()),
            pred_max=float(pred_noise.max()),
        )
        noises.append(ligand_emb_noise.cpu())
    del b, ligand_emb_noise
    return torch.stack(noises)


def sample_batch(model: PepEDiff, dataset: NoisedReceptorPeptideDataset):
    test_dataloader = DataLoader(
        dataset, batch_size=CONFIG["batch_size"], prefetch_factor=2, num_workers=16
    )
    for batch_idx, batch in enumerate(test_dataloader):
        print(f"Generating Batch {batch_idx}/{len(test_dataloader)}")
        # Sample noise and sample the lengths
        sampled = p_sample_loop(
            model=model,
            batch=batch,
            betas=dataset.alpha_beta_terms["betas"],
        )
        # Gets to size (timesteps, seq_len, n_ft)
        batch_size = len(batch.batch.unique())
        peptide_mask = batch.peptide_mask.reshape(batch_size, -1, CONFIG["max_seq_len"])
        ligand_length = [m.sum().int() for m in peptide_mask]
        trimmed_sampled = [
            sampled[:, i, :l, :].numpy() for i, l in enumerate(ligand_length)
        ]
        trimmed_sampled = [s[-1] for s in trimmed_sampled]  # extract last time step
        print(trimmed_sampled[0].shape)
        for pdb_id, sampled in zip(batch["pdb_id"], trimmed_sampled):
            if(len(os.listdir(OUTPUT_FOLDER))>0):
                curr_max = max([int(i.split("_")[-1].split(".")[0]) for i in os.listdir(OUTPUT_FOLDER)])+1
            else:
                curr_max = 0
            torch.save(sampled, os.path.join(OUTPUT_FOLDER, f"{pdb_id}_{curr_max}.pkl"))
        del sampled, ligand_length, trimmed_sampled
    return None

In [12]:
test_ds = NoisedReceptorPeptideDataset(["3Q0H_A"]*GEN_NUM, max_len=CONFIG["max_seq_len"])
test_ds[0]

Data(edge_index=[2, 4441], X=[109, 5, 3], num_nodes=109, receptor_structure=[109, 20], receptor_seq_emb=[109, 1024], pocket_mask=[109], peptide_emb=[15, 1024], peptide_mask=[128], noised_emb=[128, 1024], known_noise=[128, 1024], timestep=[1], pdb_id='3Q0H_A')

In [ ]:
seed_everything(0)
diff_model = load_model()
sample_batch(diff_model, test_ds)

In [14]:
# 4. Transfer Back to Sequence

def has_repeated_AA(s: str, threshold: float = 0.3) -> bool:
    if len(s) <= 3:
        return False
    # Calculate the threshold count
    threshold_count = len(s) * threshold
    # Create a dictionary to count occurrences of each character
    char_count = {}
    # Count occurrences of each character
    for char in s:
        if char in char_count:
            char_count[char] += 1
        else:
            char_count[char] = 1
    # Check if any character exceeds the threshold count
    for count in char_count.values():
        if count > threshold_count:
            return True
    return False

def has_consecutive_AA(s: str, threshold: float = 0.3) -> bool:
    if len(s) <= 3:
        return False
    # Calculate the threshold count
    threshold_count = len(s) * threshold
    # Initialize variables to track the current character and its consecutive count
    max_consecutive_count = 0
    current_char = ""
    current_consecutive_count = 0
    # Iterate through the string to count consecutive characters
    for char in s:
        if char == current_char:
            current_consecutive_count += 1
        else:
            current_char = char
            current_consecutive_count = 1
        # Update the max consecutive count
        if current_consecutive_count > max_consecutive_count:
            max_consecutive_count = current_consecutive_count
    # Check if the max consecutive count exceeds the threshold count
    return max_consecutive_count > threshold_count

def emb_to_seq_noise(decoder_model, tokenizer, pred_embedding):
    extra_noise_scale = 0.3
    success_flag = False
    gen_round = 0
    emb = torch.Tensor(pred_embedding).to(device).unsqueeze(0)
    while (not success_flag) and extra_noise_scale != 0:
        if gen_round != 0:
            noised_emb = emb + torch.randn_like(emb) * extra_noise_scale
        else:
            noised_emb = emb
        decoder_input_ids = tokenizer("<pad>", return_tensors="pt").input_ids
        decoder_input_ids = decoder_input_ids.to(device)
        # print(noised_emb.shape)
        # Generate the output sequence using the noised hidden states from the encoder
        output_ids = decoder_model.generate(
            input_ids=decoder_input_ids,
            encoder_outputs=(noised_emb,),
            max_length=emb.shape[1]+1,
        )
        output_seq = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        output_seq = output_seq.replace(" ", "")
        success_flag = (
            (not has_consecutive_AA(output_seq))
            and (not has_repeated_AA(output_seq))
            # and (output_seq.count("C") >= 2)
        )
        if (gen_round % 50) == 0 and gen_round != 0:
            extra_noise_scale += 0.1
            print("Increase Noise to:", extra_noise_scale)
        if gen_round >= 500:
            break
        gen_round += 1
    print(tokenizer.decode(output_ids[0], skip_special_tokens=True), gen_round)
    seq = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return seq

In [ ]:
SCALER = "./z_score_scaler.pkl"
with open(SCALER, "rb") as f:
    scaler = pickle.load(f)

In [ ]:
SAMPLED_FODLER = "./data/sample_res/sample_TIGIT"
sampled_files = sorted(os.listdir(SAMPLED_FODLER))
all_seqs = []
for f in sampled_files:
    file_path = os.path.join(SAMPLED_FODLER, f)
    emb = torch.load(file_path)
    emb = scaler.inverse_transform(emb)
    all_seqs.append(emb_to_seq_noise(emb_model, tokenizer, emb))

In [ ]:
torch.save(all_seqs, "./data/sample_res/sample_TIGIT/all_seqs.pt")